# Kafka consumer

In [ ]:
from confluent_kafka import Consumer

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test',
    'auto.offset.reset': 'earliest',
}
topic1 = 'traffic-object-raw'
topic2 = 'weather-raw'
consumer = Consumer(conf)
# consumer.subscribe(['weather-raw'])
consumer.subscribe([topic2])


try:
    while True:
        msg = consumer.poll(timeout=1.0)  # Polls for up to 1 second
        if msg is None:
            continue  # No message, go back and poll again
        if msg.error():
            print(f"Consumer error: {msg.error()}")
            continue
        print(f"Received message: {msg.value().decode('utf-8')}")
except KeyboardInterrupt:
    pass
finally:
    consumer.close()


Received message: {"image_shape": [720, 1280], "total": 3, "objects": [{"class_object": "person", "coordinates": [100, 150, 200, 300], "confidence": 0.92, "class_id": 0, "classname": "person"}, {"class_object": "bicycle", "coordinates": [250, 400, 300, 500], "confidence": 0.87, "class_id": 1, "classname": "bicycle"}, {"class_object": "bus", "coordinates": [600, 700, 650, 750], "confidence": 0.95, "class_id": 2, "classname": "bus"}], "timestamp": "2025-05-02 13:13:56", "cam_id": "56de42f611f398ec0c481289", "img": "datalake/raw/traffic/56de42f611f398ec0c481289/2025-05-02/13-13-56.jpg"}
Received message: {"image_shape": [720, 1280], "total": 3, "objects": [{"class_object": "person", "coordinates": [100, 150, 200, 300], "confidence": 0.92, "class_id": 0, "classname": "person"}, {"class_object": "bicycle", "coordinates": [250, 400, 300, 500], "confidence": 0.87, "class_id": 1, "classname": "bicycle"}, {"class_object": "bus", "coordinates": [600, 700, 650, 750], "confidence": 0.95, "class_id

In [17]:
from confluent_kafka import Consumer, TopicPartition
from datetime import datetime, timedelta

topic = 'weather-raw'
interval = 15 # last 15 min

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test',
    'enable.auto.commit': False,
    'auto.offset.reset': 'earliest'  # Only used if no committed offsets exist
}

consumer = Consumer(conf)
metadata = consumer.list_topics(topic)
partitions = metadata.topics[topic].partitions.keys()
# topic_partitions = [TopicPartition(topic, p) for p in partitions]

# timestamp for last 15 mins
timestamp_ms = int((datetime.now() - timedelta(minutes=interval)).timestamp() * 1000)
timestamp_partitions = [TopicPartition(topic, p, timestamp_ms) for p in partitions]
offsets = consumer.offsets_for_times(timestamp_partitions, timeout=10.0)

# Assign and seek to the correct offsets
valid_offsets = []
for tp in offsets:
    if tp.offset != -1:  # Offset -1 means no data available for that timestamp
        valid_offsets.append(TopicPartition(tp.topic, tp.partition, tp.offset))

consumer.assign(valid_offsets)

# Start consuming
while True:
    msg = consumer.poll(1.0)  # Waits up to 1 second for new messages
    if msg is None:
        break  # Stop when nothing is returned
    if msg.error():
        print(f"Error: {msg.error()}")
        continue
    print(msg.value().decode())


%6|1746339325.501|FAIL|rdkafka#consumer-17| [thrd:localhost:9092/0]: localhost:9092/0: Disconnected (after 4230569ms in state UP)
%6|1746339325.558|FAIL|rdkafka#consumer-17| [thrd:localhost:9092/0]: localhost:9092/0: Disconnected while requesting ApiVersion: might be caused by incorrect security.protocol configuration (connecting to a SSL listener?) or broker version is < 0.10 (see api.version.request) (after 53ms in state APIVERSION_QUERY)
%6|1746339325.583|FAIL|rdkafka#consumer-17| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Disconnected while requesting ApiVersion: might be caused by incorrect security.protocol configuration (connecting to a SSL listener?) or broker version is < 0.10 (see api.version.request) (after 22ms in state APIVERSION_QUERY)
%6|1746339325.785|FAIL|rdkafka#consumer-17| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Disconnected while requesting ApiVersion: might be caused by incorrect security.protocol configuration (connecting to a S

# Google cloud Storage

In [2]:
from dotenv import load_dotenv
import os
import json
# Imports the Google Cloud client library
from google.cloud import storage
from google.oauth2 import service_account
import logging

class GoogleStorageClient:
    def __init__(self, bucket_name):
        self.bucket_name = bucket_name
        self.client = self.__init_config()
        self.bucket = self.client.bucket(bucket_name)
    

    def __init_config(self):
        load_dotenv() 
        credential_info = json.loads(os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
        credentials = service_account.Credentials.from_service_account_info(credential_info)
        client = storage.Client(credentials=credentials, project=credential_info.get("project_id"))
        return client


    def upload_file(self, file_path, destination_blob_name):
        blob = self.bucket.blob(destination_blob_name)
        blob.upload_from_filename(file_path)
        logging.info(f"Uploaded {file_path} to gs://{self.bucket_name}/{destination_blob_name}")


    def download_file(self, source_blob_name, destination_file_name):
        blob = self.bucket.blob(source_blob_name)
        blob.download_to_filename(destination_file_name)
        logging.info(f"Downloaded gs://{self.bucket_name}/{source_blob_name} to {destination_file_name}")
    

    def delete_file(self, blob_name):
        blob = self.bucket.blob(blob_name)
        try:
            blob.delete()
            logging.info(f"Deleted gs://{self.bucket_name}/{blob_name}")
        except Exception as e:
            logging.error(f"Error deleting gs://{self.bucket_name}/{blob_name}: {e}")
    
    
    def is_file_exists(self, blob_name):
        blob = self.bucket.blob(blob_name)
        return blob.exists()

    def list_blobs(self, prefix=None):
        blobs = self.client.list_blobs(self.bucket_name, prefix=prefix)
        return blobs

In [4]:
gcs_client = GoogleStorageClient('traffic_flow_thesis')

gcs_client.upload_file("producer.py", "test/producer.py")
gcs_client.download_file("test/producer.py", "producer_downloaded.py")
gcs_client.delete_file("test/producer.py")

In [1]:
from dotenv import load_dotenv
import os
import json
# Imports the Google Cloud client library
from google.cloud import storage
from google.oauth2 import service_account


load_dotenv()
credential_info = json.loads(os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_info(credential_info)
client = storage.Client(credentials=credentials, project=credential_info.get("project_id"))

# # The name for the new bucket
for bucket in client.list_buckets():
    print(bucket.name)

traffic_flow_thesis


# Spark GCS

In [4]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from contextlib import contextmanager
import json
import os
import pandas as pd
# import logging

packages = [
  "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"

]
conf = (SparkConf().setAppName("GCS-READER")
    .set("spark.executor.memory", "2g")
    # .set("spark.sql.repl.eagerEval.enabled", True)
    .set("spark.jars", ",".join(packages))
    .setMaster("local[*]")
    )

@contextmanager
def SparkIO(conf: SparkConf = conf, gcs: bool = False):
    app_name = conf.get("spark.app.name")
    master = conf.get("spark.master")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    print(f'Create SparkSession app {app_name} with {master} mode')


    try:
        if gcs:
            service_account_path = "/tmp/service_account.json"
            with open(service_account_path, "w") as f:
                json.dump(json.loads(os.getenv('GOOGLE_APPLICATION_CREDENTIALS')), f)

            spark._jsc.hadoopConfiguration().set("fs.gs.auth.service.account.json.keyfile", service_account_path)
            # spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.enable", "true")
            # spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", gg_service_path)

        yield spark
    except Exception:
        raise Exception
    finally:
        print(f'Stop SparkSession app {app_name}')

        if os.path.exists(service_account_path):
            os.remove(service_account_path)

        spark.stop()

In [6]:
bucket = "traffic_flow_thesis"

with SparkIO(gcs=True) as spark:

  data = spark.read \
  .json(f"gs://{bucket}/raw_event/traffic-object-raw/year=2025") # read all of year 2025

  # f"gs://{bucket}/raw_event/weather-raw/year=2025/month=05/day=10"

  data.show()

  df = data.toPandas() # convert to pandas

Create SparkSession app GCS-READER with local[*] mode


+--------------------+-----------+--------------------+--------------------+-------------------+-----+-----+---+----+
|              cam_id|image_shape|                 img|             objects|          timestamp|total|month|day|hour|
+--------------------+-----------+--------------------+--------------------+-------------------+-----+-----+---+----+
|56de42f611f398ec0...| [288, 512]|traffic_flow_thes...|                  []|2025-05-10 00:05:02|    0|    5| 10|  13|
|56de42f611f398ec0...| [177, 284]|traffic_flow_thes...|                  []|2025-05-10 13:48:44|    0|    5| 10|  13|
|56de42f611f398ec0...| [177, 284]|traffic_flow_thes...|                  []|2025-05-10 13:48:44|    0|    5| 10|  13|
|56de42f611f398ec0...| [288, 512]|traffic_flow_thes...|[{2, car, car, 0....|2025-05-10 13:48:44|   18|    5| 10|  13|
|58b5752e17139d001...| [450, 800]|traffic_flow_thes...|[{2, car, car, 0....|2025-05-10 13:48:44|   10|    5| 10|  13|
|56de42f611f398ec0...| [177, 284]|traffic_flow_thes...| 

Stop SparkSession app GCS-READER
